In [1]:
import pandas as pd
from uuid import uuid4
from transformers import AutoTokenizer
from math import ceil

tokenizer = AutoTokenizer.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-31-8B-Instruct-mntp"
)

In [14]:
df = pd.read_parquet('/raid/deallab/SF_RAG_Data/ASQA/train.parquet')

In [10]:
df.head()

,ambiguous_question,qa_pairs,wikipages,annotations,sample_id
0,When does the new bunk'd come out?,"[{'context': 'No context provided', 'question'...","[{'title': 'List of Bunk'd episodes', 'url': '...","[{'knowledge': [{'content': None, 'wikipage': ...",-5742327688291876861
1,Who won the 2016 ncaa football national champi...,[{'context': 'The 13–1 Alabama Crimson Tide wo...,[{'title': '2015 College Football Playoff Nati...,[{'knowledge': [{'content': 'The 13–1 Alabama ...,-3582047784487750233
2,When was the last time the death penalty was u...,"[{'context': 'No context provided', 'question'...",[{'title': 'Capital punishment in Pennsylvania...,[{'knowledge': [{'content': 'Heidnik was execu...,6811938153834854976
3,Where will failure of the left ventricle cause...,"[{'context': '""Backward"" failure of the left v...","[{'title': 'Heart failure', 'url': 'https://en...","[{'knowledge': [{'content': '""Backward"" failur...",1700733897006170137
4,Who won the war between ethiopia and italy?,[{'context': 'The First Italo-Ethiopian War wa...,"[{'title': 'Second Italo-Ethiopian War', 'url'...",[{'knowledge': [{'content': 'Italian defeat ca...,142117929623619257


In [11]:
for col in df.columns:
    print(col,':')
    print(df.loc[1, col], '\n')

ambiguous_question :
Who won the 2016 ncaa football national championship? 

qa_pairs :
[{'context': "The 13–1 Alabama Crimson Tide won the game, holding off the undefeated Clemson Tigers 45–40 in the fourth quarter. Accompanied by a talented receiving corps, Clemson's Heisman Finalist quarterback Deshaun Watson had a historic performance, setting the record for most total yards in national championship game history, with 478 yards (405 passing / 73 rushing) against the nation's third-ranked defense in Alabama, breaking the record previously set by Vince Young in the 2006 Rose Bowl. Following the game, the AP Poll also named Alabama as its top team of the season, giving Alabama their fourth title in seven seasons. Both Clemson and Alabama finished the season 14–1.", 'question': "Who won the 2016 season's ncaa football national championship?", 'short_answers': array(['Clemson Tigers', '2016 Clemson Tigers football team',
        '2016 Clemson Tigers football', 'the Tigers', 'Clemson',
 

In [18]:
# create embedding document dataset.
evidence_df = pd.DataFrame(columns=['id','sample_ids','title', 'url', 'text'])

#creating question evidence pairs for retrival training
qe_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'evidence_id'])

for idx, row in df.iterrows():
    if idx == 2: break
    evidences = row['wikipages']
    for evidence in evidences:
        url = evidence['url']
        page = requests.get('https://en.wikipedia.org/wiki/2015%20College%20Football%20Playoff%20National%20Championship')
        
        # Create a BeautifulSoup object
        soup = BeautifulSoup(page.text, 'html.parser')
        # get title
        title = soup.find(class_='mw-page-title-main').contents[0]
        
        #extract content
        content = soup.find(class_='mw-content-ltr')
        parsed_doc = parse_document(content)
        
        # chunk document
        documents = []
        
        for par in re.split(r'(?=\n#{1,4})', parsed_doc):
            tokenized_par = tokenizer.encode(par, add_special_tokens = False)
            length = len(tokenized_par)
            if len(documents[-1]) + length < 1010:
                documents[-1].extend(tokenized_par)
            elif length > 1010:
                begin = 0 
                while begin < length:
                    if begin + 1010 >= length:
                        documents.append(tokenized_par[begin:])
                        break
                    documents.append(tokenized_par[begin:begin + 1010])
                    begin += 810
            else:
                documents.append([par])
                
            for doc in documents:
                doc_text = tokenizer.decode(doc)
                evidence_df.loc[len(eidence_df)] = [uuid4(),]

https://en.wikipedia.org/wiki/List%20of%20Bunk%27d%20episodes
https://en.wikipedia.org/wiki/2015%20College%20Football%20Playoff%20National%20Championship
https://en.wikipedia.org/wiki/2016%20College%20Football%20Playoff%20National%20Championship
https://en.wikipedia.org/wiki/2017%20College%20Football%20Playoff%20National%20Championship


In [19]:
uuid4()

UUID('b9d9a0c0-b28f-4b42-8ad8-77c704df3526')

In [2]:
import requests
from bs4 import BeautifulSoup
import re


page = requests.get('https://en.wikipedia.org/wiki/2015%20College%20Football%20Playoff%20National%20Championship')

# Create a BeautifulSoup object
soup = BeautifulSoup(page.text, 'html.parser')
title = soup.find(class_='mw-page-title-main').contents[0]
content = soup.find(class_='mw-content-ltr')

# with open('./temp.html', 'w') as f:
#     f.write(str(content))

In [3]:
def get_table(table):
    table_text = []
    for i, tr in enumerate(table.find('tbody').findChildren("tr" , recursive=False)):
        tr_text = tr.get_text()
        tr_text = re.sub(r'\n+',';',tr_text).strip(';')
        if not tr_text: continue
        if table_text == []:
            tr_text = '\n#### Table: ' + tr_text
        table_text.append(tr_text)
    return '\n'.join(table_text)

def get_p(par):
    p_text = par.get_text()
    p_text = p_text.replace('\n','')
    return p_text

def get_h(heading):
    h = heading.find(['h1', 'h2', 'h3','h4'])
    heading_type = int(re.search(r'<h(\d)', str(h)).group(1))
    h_text ='\n' + ' '.join(['#'*heading_type,h.get_text()])
    return h_text

def get_ul(ul):
    list_text = []
    for li in ul.find_all('li'):
        list_text.append('* ' + li.get_text())
    return '\n'.join(list_text)

def get_ol(ol):
    list_text = []
    for i, li in enumerate(ol.find_all('li')):
        if li.get_text():
            list_text.append(' '.join([str(i+1),li.get_text().replace('\n','')]))
    return '\n'.join(list_text)
        

def parse_document(doc):
    content = doc.find_all(['div', 'p', 'table', 'ul', 'ol'])
    document  = []
    for cont in content:
        #stop condition
        if cont.name == 'div' and cont.find(['h1', 'h2', 'h3','h4'], id=['See_also', 'References']):
            break
        
        #get headining
        if cont.name == 'div' and cont.has_attr('class') and  'mw-heading' in cont['class']:
            document.append(get_h(cont))
        # get par
        elif cont.name == 'p':
            par = get_p(cont)
            if par:
                document.append(par)
        #get ul
        elif cont.name == 'ul':
            document.append(get_ul(cont))
        #get ol
        elif cont.name == 'ol':
            document.append(get_ol(cont))
        #get table
        elif cont.name == 'table':
            if cont.has_attr('class') and 'metadata' in cont['class']: continue
            document.append(get_table(cont))
        # explore div
        elif cont.name == 'div':
            document.append(parse_document(cont))

    return '\n'.join(document).strip('\n')



'#### Table: 2015  College Football Playoff National Championship presented by AT&T\nInaugural College Football Playoff National Championship\nOhio State Buckeyes;Oregon Ducks;(13–1);(13–1);Big Ten;Pac-12;42;20;Head\xa0coach:\xa0Urban Meyer;Head\xa0coach:\xa0Mark Helfrich;APCoachesCFP;544;APCoachesCFP;332\n1234;Total;Ohio State;147714;42;Oregon;73100;20\nDateJanuary 12, 2015\nSeason2014\nStadiumAT&T Stadium\nLocationArlington, Texas\nMVPOffensive: #15 RB Ezekiel Elliott, So. Ohio StateDefensive: #23 S Tyvis Powell, So. Ohio State\nFavoriteOregon by 7[1][2]\nNational anthemLady Antebellum[3]\nRefereeGreg Burks (Big 12)\nAttendance85,689\nUnited States TV coverage\nNetworkESPN[4][5]\nAnnouncersChris Fowler, Kirk Herbstreit, Heather Cox and Tom Rinaldi (ESPN)Eduardo Varela and Pablo Viruega (ESPN Deportes)Mike Tirico, Todd Blackledge, Holly Rowe and Joe Schad (ESPN Radio)\nNielsen ratings18.9 (33.4 million viewers)\n College Football Playoff National Championship ; \xa0 ; 2016 >\xa0 ;Coll

In [12]:
documents = [[]]

for par in re.split(r'(?=\n#{1,4})', parsed_doc):
    tokenized_par = tokenizer.encode(par, add_special_tokens = False)
    length = len(tokenized_par)
    if len(documents[-1]) + length < 1010:
        documents[-1].extend(tokenized_par)
    elif length > 1010:
        begin = 0 
        while begin < length:
            if begin + 1010 >= length:
                documents.append(tokenized_par[begin:])
                break
            documents.append(tokenized_par[begin:begin + 1010])
            begin += 810
    else:
        documents.append([par])

for doc in documents:
    print(tokenizer.decode(doc))

#### Table: 2015  College Football Playoff National Championship presented by AT&T
Inaugural College Football Playoff National Championship
Ohio State Buckeyes;Oregon Ducks;(13–1);(13–1);Big Ten;Pac-12;42;20;Head coach: Urban Meyer;Head coach: Mark Helfrich;APCoachesCFP;544;APCoachesCFP;332
1234;Total;Ohio State;147714;42;Oregon;73100;20
DateJanuary 12, 2015
Season2014
StadiumAT&T Stadium
LocationArlington, Texas
MVPOffensive: #15 RB Ezekiel Elliott, So. Ohio StateDefensive: #23 S Tyvis Powell, So. Ohio State
FavoriteOregon by 7[1][2]
National anthemLady Antebellum[3]
RefereeGreg Burks (Big 12)
Attendance85,689
United States TV coverage
NetworkESPN[4][5]
AnnouncersChris Fowler, Kirk Herbstreit, Heather Cox and Tom Rinaldi (ESPN)Eduardo Varela and Pablo Viruega (ESPN Deportes)Mike Tirico, Todd Blackledge, Holly Rowe and Joe Schad (ESPN Radio)
Nielsen ratings18.9 (33.4 million viewers)
 College Football Playoff National Championship ;   ; 2016 >  ;College Football Championship Game;  < 2

TypeError: argument 'ids': 'str' object cannot be interpreted as an integer